# Predicting whether a loan will default (Charged Off) vs be full repaid, using loan application and performance data from LendingClub 

Lenders lose money when borrowers fail to repay, so estimating the probability of default (PD) before approving a loan is a core part of credit risk management. This project uses historical LendingClub loan data (2007–2019) to predict whether a loan will be Fully Paid or Charged Off, based on information available when the loan was issued, such as loan amount, interest rate, income, debt-to-income ratio and credit history. Because defaults are much rarer than repayments, the project addresses class imbalance and evaluates models with precision, recall and ROC-AUC instead of accuracy. It also considers the different costs of approving a loan that defaults versus rejecting one that would have been repaid, to choose a decision threshold a lender could use in practice.

## Intial Setup & inspect structure
Target: default = 1 for Charged Off and Default, Fully Paid = 0. Excluded loans with no final outcome yet (Current, Late, In Grace Period) and early loans issued under a superseded credit policy. Result: 1,385,902 loans, about 19.8% defaulted.

In [ ]:
# Run this first after every kernel restart
import gc
import pandas as pd

In [1]:
df_raw = pd.read_csv("../data/raw/appl_accepted_20072019Q3.csv", low_memory=False)
print(df_raw['loan_status'].value_counts())

df = df_raw[df_raw['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()

del df_raw
gc.collect()

print(df.shape)
df.to_parquet("../data/processed/loans_resolved.parquet", index=False)

KeyboardInterrupt: 

In [1]:
df = pd.read_parquet("../data/processed/loans_resolved.parquet")
df.info(verbose=True, show_counts=True)

# 0 = Fully Paid, 1 = Charged Off or Default
df['default'] = df['loan_status'].isin(['Charged Off', 'Default']).astype(int)
print(df['default'].value_counts(normalize=True))

NameError: name 'pd' is not defined

### Missingness check

In [ ]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
print(missing[missing > 0].to_string())

member_id                                     100.000000
next_pymnt_d                                   99.996753
orig_projected_additional_accrued_interest     99.728624
hardship_payoff_balance_amount                 99.584675
hardship_loan_status                           99.584675
hardship_type                                  99.584675
hardship_reason                                99.584675
hardship_amount                                99.584675
hardship_status                                99.584675
hardship_start_date                            99.584675
hardship_length                                99.584675
hardship_dpd                                   99.584675
hardship_last_payment_amount                   99.584675
hardship_end_date                              99.584675
payment_plan_start_date                        99.584675
deferral_term                                  99.584675
sec_app_mths_since_last_major_derog            99.412946
sec_app_revol_util             

In [ ]:
# issue_d has two formats: 'Dec-2018' (most rows) and 'Mar-19' (2019 loans)
d = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['issue_d'] = d.fillna(pd.to_datetime(df['issue_d'], format='%b-%y', errors='coerce'))
df['issue_year'] = df['issue_d'].dt.year
print('Unparsed dates:', df['issue_d'].isna().sum())

print(df.groupby('issue_year')['open_acc_6m'].apply(lambda s: s.isnull().mean()).round(2))

Unparsed dates: 0
issue_year
2007    1.00
2008    1.00
2009    1.00
2010    1.00
2011    1.00
2012    1.00
2013    1.00
2014    1.00
2015    0.95
2016    0.00
2017    0.00
2018    0.00
2019    0.00
Name: open_acc_6m, dtype: float64


### Drop columns

In [ ]:
# Group 2: "has ever had X" flags, only where blank really means "never"
flag_cols = ['mths_since_last_delinq', 'mths_since_last_record']
for c in flag_cols:
    df['has_' + c.replace('mths_since_', '')] = df[c].notnull().astype(int)

# Not collected before 2012, so blank there doesn't mean "never". Drop for v1
uncollected = ['mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq']

# Group 3: block only collected from late 2015
block_58 = ['open_acc_6m', 'inq_last_12m', 'total_cu_tl', 'open_il_12m', 'open_il_24m', 'open_rv_12m',
            'open_rv_24m', 'max_bal_bc', 'total_bal_il', 'open_act_il', 'inq_fi', 'all_util', 'il_util',
            'mths_since_rcnt_il']

# Group 1: empty, identifiers/free text, joint-only, hardship/settlement
drop_cols = (['member_id', 'id', 'url', 'desc', 'emp_title', 'title', 'zip_code', 'next_pymnt_d',
              'orig_projected_additional_accrued_interest', 'deferral_term', 'payment_plan_start_date',
              'revol_bal_joint', 'verification_status_joint', 'dti_joint', 'annual_inc_joint',
              'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_date']
             + [c for c in df.columns if c.startswith(('hardship_', 'settlement_', 'sec_app_'))])

# Single value across every row
constant = ['policy_code', 'pymnt_plan', 'hardship_flag']

# Post-outcome leakage: describes what happened after approval
leakage = ['out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
           'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
           'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low']

to_drop = list(dict.fromkeys(flag_cols + uncollected + block_58 + drop_cols + constant + leakage))
print('Names not found:', [c for c in to_drop if c not in df.columns])

df = df.drop(columns=to_drop, errors='ignore')
print(df.shape)

Names not found: []
(1385902, 72)


In [ ]:
# Stored as text; convert so median imputation works later
for c in ['int_rate', 'revol_util']:
    df[c] = pd.to_numeric(df[c].astype(str).str.rstrip('%'), errors='coerce')

# Any remaining columns with a single value?
print('Constant columns:', [c for c in df.columns if df[c].nunique(dropna=False) <= 1])

print(df.columns.tolist())

Constant columns: []
['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',

### Missingness decisions
- **Flags:** `mths_since_last_delinq` and `mths_since_last_record` became 0/1 `has_*` flags, since a blank means the borrower never had that event.
- **Not-yet-collected columns:** `mths_since_last_major_derog`, `mths_since_recent_bc_dlq` and `mths_since_recent_revol_delinq` are 100% blank for 2007–2011 loans because LendingClub hadn't started collecting them, so a blank doesn't mean "never". Dropped for v1.
- **58% block:** 14 account-activity columns (`open_acc_6m`, `il_util`, etc.) are blank for every loan before late 2015. Dropped rather than imputed.
- **Leakage:** payment, recovery and settlement columns (`total_pymnt`, `recoveries`, `debt_settlement_flag`, etc.) describe what happened after approval, so they were removed.
- **Small gaps:** about 4.9% of loans (mostly pre-2012) are missing the `num_*`, `tot_*` and `mo_sin_*` fields. These will be imputed with the median inside the pipeline, after the train/test split, with missing-value indicators.

### Duplicate check

In [ ]:
# Duplicate loans: rows identical on every column (id was dropped above; it had no repeats either)
print('Duplicate rows:', df.duplicated().sum())

Duplicate rows: 0


### Categorical variables

In [ ]:
# funded_amnt_inv is settled after listing (mild leakage); funded_amnt nearly duplicates loan_amnt
df = df.drop(columns=['funded_amnt', 'funded_amnt_inv'])

# term: ' 36 months' -> 36
df['term'] = df['term'].str.extract(r'(\d+)', expand=False).astype(int)

# emp_length: '< 1 year' -> 0, '10+ years' -> 10, blanks stay missing (imputed in the pipeline)
df['emp_length'] = (df['emp_length'].replace('< 1 year', '0 years')
                    .str.extract(r'(\d+)', expand=False).astype(float))

# sub_grade: A1 -> 1 ... G5 -> 35 (ordinal, finer than grade)
# grade is kept as letters for EDA and dropped before modelling
df['sub_grade_num'] = (df['sub_grade'].str[0].map({g: i for i, g in enumerate('ABCDEFG')}) * 5
                       + df['sub_grade'].str[1].astype(int))
df = df.drop(columns='sub_grade')

# home_ownership: merge the tiny ANY / NONE categories into OTHER
df['home_ownership'] = df['home_ownership'].replace({'ANY': 'OTHER', 'NONE': 'OTHER'})

# earliest_cr_line -> credit history length in months at issue date
# Same two formats as issue_d; '%b-%y' reads 'Mar-65' as 2065, so pull any future date back 100 years
ecl = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
ecl = ecl.fillna(pd.to_datetime(df['earliest_cr_line'], format='%b-%y', errors='coerce'))
ecl = ecl.where(ecl <= df['issue_d'], ecl - pd.DateOffset(years=100))
df['credit_history_months'] = ((df['issue_d'].dt.year - ecl.dt.year) * 12
                               + (df['issue_d'].dt.month - ecl.dt.month))
df = df.drop(columns='earliest_cr_line')

print('Unparsed earliest_cr_line:', ecl.isna().sum())
print(df[['term', 'emp_length', 'sub_grade_num', 'credit_history_months']].describe().round(1))
print(df.select_dtypes('object').nunique())

Unparsed earliest_cr_line: 0
            term  emp_length  sub_grade_num  credit_history_months
count  1385902.0   1303611.0      1385902.0              1385902.0
mean        41.8         6.0           11.6                  195.0
std         10.3         3.7            6.4                   90.1
min         36.0         0.0            1.0                   36.0
25%         36.0         2.0            7.0                  135.0
50%         36.0         6.0           11.0                  177.0
75%         36.0        10.0           15.0                  240.0
max         60.0        10.0           35.0                  999.0
grade                   7
home_ownership          4
verification_status     3
loan_status             3
purpose                14
addr_state             51
initial_list_status     2
application_type        2
dtype: int64


### Categorical encoding plan
- **Converted to numbers here:** `term` (36/60), `emp_length` (0–10, blanks imputed later), `sub_grade` → `sub_grade_num` (1–35, ordinal) and `earliest_cr_line` → `credit_history_months`.
- **One-hot in the pipeline (Phase 3):** `home_ownership` (ANY/NONE merged into OTHER), `verification_status`, `purpose`, `initial_list_status`, `application_type` and `addr_state`.
- **Kept for EDA, dropped before modelling:** `grade` (overlaps with `sub_grade_num`), `loan_status`, `issue_d`, `issue_year`.
- **Also dropped:** `funded_amnt_inv` (settled after listing) and `funded_amnt` (near-duplicate of `loan_amnt`).
- **Duplicates:** none found.

### Save cleaned dataset

In [ ]:
assert 'default' in df.columns, "'default' is missing: run the cell that loads loans_resolved.parquet first"
df.to_parquet("../data/processed/loans_clean.parquet", index=False)
print(df.shape)

(1385902, 70)


## Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import LinearSegmentedColormap

# Start EDA from the saved clean file, so Phase 2 doesn't depend on rerunning Phase 1
df = pd.read_parquet("../data/processed/loans_clean.parquet")

BLUE, GREY = '#257de0', '#666666'
sns.set_theme(style='whitegrid', rc={'grid.color': '#e6e6e6', 'axes.edgecolor': '#cccccc'})
base_rate = df['default'].mean()
print(df.shape)

KeyError: 'default'

### Check class balance of target
80.23% of loans are fully paid and 19.77% of loans are charged off (default).

In [ ]:
# default was created in Phase 1: 0 = Fully Paid, 1 = Charged Off or Default
counts = df['default'].value_counts()
print(pd.DataFrame({'loans': counts, 'share': (counts / len(df)).round(4)}))

### Univariate analysis

In [ ]:
key_num = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'credit_history_months']
print(df[key_num].describe().round(1).T)

# describe() skips text columns, so count the categorical ones separately
print(df['grade'].value_counts().sort_index())
print(df['term'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, key_num):
    s = df[col].dropna()
    s = s[s <= s.quantile(0.99)]  # trim the top 1% so long tails don't squash the plot
    ax.hist(s, bins=50, color=BLUE, edgecolor='white', linewidth=0.5)
    ax.set_title(col)
    ax.set_ylabel('loans')
fig.suptitle('Distributions of key numeric features (top 1% trimmed)')
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={'width_ratios': [3, 1]})
for ax, col in zip(axes, ['grade', 'term']):
    vc = df[col].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color=BLUE)
    ax.set_title(f'Loans by {col}')
    ax.set_ylabel('loans')
fig.tight_layout()
plt.show()

In [ ]:
# Values that look like data-entry errors or extreme outliers
print('annual_inc = 0:            ', (df['annual_inc'] == 0).sum())
print('annual_inc > $1M:          ', (df['annual_inc'] > 1e6).sum())
print('dti < 0 or > 100:          ', ((df['dti'] < 0) | (df['dti'] > 100)).sum())
print('credit history > 60 years: ', (df['credit_history_months'] > 720).sum())

### Bivariate analysis: default rate by feature
Each bar is the share of loans in that group that defaulted. The dashed line is the overall rate (19.8%), so bars above it are riskier than average.

In [ ]:
def rate_by(key):
    """Default rate and loan count for each value of key."""
    return df.groupby(key, observed=True)['default'].agg(rate='mean', loans='size')

def bar_rate(ax, r, title):
    ax.bar(r.index.astype(str), r['rate'], color=BLUE)
    ax.axhline(base_rate, color=GREY, ls='--', lw=1)  # dashed line = overall default rate
    ax.set_title(title)
    ax.set_ylabel('default rate')
    ax.yaxis.set_major_formatter(PercentFormatter(1))

emp = df['emp_length'].map(lambda v: 'missing' if pd.isna(v) else str(int(v)))
emp_order = [str(i) for i in range(11)] + ['missing']
purpose = rate_by(df['purpose']).sort_values('rate')

fig, axes = plt.subplots(3, 2, figsize=(14, 13))
bar_rate(axes[0, 0], rate_by(df['grade']), 'Default rate by grade')
bar_rate(axes[0, 1], rate_by(df['term']), 'Default rate by term (months)')
bar_rate(axes[1, 0], rate_by(df['home_ownership']).sort_values('rate'), 'Default rate by home ownership')
bar_rate(axes[1, 1], rate_by(df['verification_status']).sort_values('rate'), 'Default rate by income verification')
bar_rate(axes[2, 0], rate_by(emp).reindex(emp_order), 'Default rate by employment length (years)')
bar_rate(axes[2, 1], purpose, 'Default rate by loan purpose')
plt.setp(axes[2, 1].get_xticklabels(), rotation=60, ha='right')
fig.tight_layout()
plt.show()

print(purpose)  # some purposes are tiny (e.g. educational), so check counts before trusting their rates

In [ ]:
# Numeric features: split into 10 equal-sized groups (deciles) and compare default rates
num_cols = ['int_rate', 'annual_inc', 'dti', 'loan_amnt', 'fico_range_low', 'credit_history_months']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_cols):
    decile = (pd.qcut(df[col], 10, labels=False, duplicates='drop') + 1).astype('Int64')
    bar_rate(ax, rate_by(decile), f'Default rate by {col} decile')
    ax.set_xlabel('decile (1 = lowest)')
fig.tight_layout()
plt.show()

In [ ]:
by_year = rate_by(df['issue_year'])
fig, ax = plt.subplots(figsize=(10, 3.5))
bar_rate(ax, by_year, 'Default rate by issue year')
plt.show()
print(by_year)

**Caution on issue year:** the low default rate for 2018–2019 is not an improvement in borrowers. The dataset ends in 2019 Q3 and only resolved loans are kept, so recent loans had less time to finish. The ones that did finish are mostly early payoffs. This matters for Phase 3: a test set made only of the newest loans would look misleadingly safe.

### Correlation check

In [ ]:
num = df.select_dtypes('number').drop(columns=['default', 'issue_year'])
corr = num.corr()

# Feature pairs that carry almost the same information
upper = corr.abs().where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(ascending=False)
print('Pairs with |r| > 0.8:')
print(pairs[pairs > 0.8].round(2).to_string())

print('\nCorrelation with default (top 15 by strength):')
print(num.corrwith(df['default']).sort_values(key=abs, ascending=False).head(15).round(3).to_string())

heat_cols = ['loan_amnt', 'installment', 'int_rate', 'sub_grade_num', 'term', 'annual_inc', 'dti',
             'fico_range_low', 'fico_range_high', 'revol_util', 'bc_util', 'open_acc', 'num_sats',
             'credit_history_months', 'mo_sin_old_rev_tl_op', 'mort_acc']
cmap = LinearSegmentedColormap.from_list('div', ['#f47230', '#efefef', '#257de0'])  # orange = negative, blue = positive
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr.loc[heat_cols, heat_cols], cmap=cmap, vmin=-1, vmax=1, annot=True, fmt='.2f',
            annot_kws={'size': 7}, square=True, linewidths=1, linecolor='white', ax=ax)
ax.set_title('Correlation between key numeric features')
fig.tight_layout()
plt.show()

### Multicollinearity decisions (apply in Phase 3)
These pairs carry nearly the same information. Logistic regression gets unstable coefficients when both are kept, so keep one of each. Tree models are less affected.
- `fico_range_high` duplicates `fico_range_low` (r = 1.00). Keep `fico_range_low`.
- `num_sats` duplicates `open_acc` (r = 1.00). Keep `open_acc`.
- `num_rev_tl_bal_gt_0` duplicates `num_actv_rev_tl` (r = 0.98). Keep `num_actv_rev_tl`.
- `int_rate` and `sub_grade_num` (r = 0.97): LendingClub sets the rate from the sub-grade. Keep one for logistic regression.
- `tot_hi_cred_lim` and `tot_cur_bal` (r = 0.97). Keep one.
- `installment` is calculated from `loan_amnt`, `int_rate` and `term` (r = 0.95 with `loan_amnt`). Drop it.
- `mo_sin_old_rev_tl_op` and `credit_history_months` (r = 0.92). Keep `credit_history_months`.

### EDA findings
*(Draft from the outputs above; rewrite in your own words.)*
1. **LendingClub's grade already captures most of the risk.** Default rises steadily from 6.1% for grade A to 49.9% for grade G, and from 4.9% to 39.4% across interest-rate deciles. The model has to beat a strong signal the lender already priced in.
2. **60-month loans default about twice as often** as 36-month loans (32.0% vs 15.9%).
3. **Surprise: verified income means higher risk, not lower.** Verified borrowers default at 23.8% vs 14.6% for unverified ones. LendingClub chose whom to verify, and it verified riskier applicants more often.
4. **Missing employment length is a risk signal.** Borrowers with no `emp_length` default at 26.5%, vs about 19–20% for every reported length. This supports keeping missing-value indicators when imputing.
5. **Income matters less than expected.** The lowest-income decile defaults at 23.9% vs 15.0% for the highest, a much smaller spread than grade or term.